In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Paths

In [2]:
BASE_DIR = "/content/drive/MyDrive/genai_project"

CLEAN_DIR = f"{BASE_DIR}/data/synthetic_clean"
DEGRADED_DIR = f"{BASE_DIR}/data/synthetic_degraded"
REAL_DIR = f"{BASE_DIR}/data/real"

TARGET_DIR = f"{BASE_DIR}/data/split"
REAL_TEST_DIR = f"{BASE_DIR}/data/real_test"

Reset

In [3]:
import shutil, os

if os.path.exists(TARGET_DIR):
    shutil.rmtree(TARGET_DIR)

if os.path.exists(REAL_TEST_DIR):
    shutil.rmtree(REAL_TEST_DIR)

print("Reset folders")

Reset folders


Collect function

In [4]:
def collect_images(base_path):
    data = {}

    for cls in os.listdir(base_path):
        class_path = os.path.join(base_path, cls)
        if not os.path.isdir(class_path):
            continue

        imgs = [os.path.join(class_path, f) for f in os.listdir(class_path)]
        data[cls] = imgs

    return data

Load data

In [5]:
import random
random.seed(42)

clean_data = collect_images(CLEAN_DIR)
degraded_data = collect_images(DEGRADED_DIR)
real_data = collect_images(REAL_DIR)

create train dataset

In [6]:
data = {}

MAX_PER_CLASS = 2000

for cls in degraded_data:

    aug   = degraded_data.get(cls, [])
    clean = clean_data.get(cls, [])

    random.shuffle(aug)
    random.shuffle(clean)

    # כמה לקחת
    n_total = min(MAX_PER_CLASS, len(aug) + len(clean))

    n_aug   = int(0.7 * n_total)
    n_clean = int(0.3 * n_total)

    selected = aug[:n_aug] + clean[:n_clean]

    random.shuffle(selected)

    data[cls] = selected

Split synthetic (train/val/test)

In [7]:
from tqdm import tqdm
import os
import shutil

train_ratio = 0.7
val_ratio = 0.15

split_counts = {}

for cls, images in tqdm(data.items(), desc="Classes"):

    random.shuffle(images)

    n = len(images)
    n_train = int(train_ratio * n)
    n_val = int(val_ratio * n)

    splits = {
        "train": images[:n_train],
        "val": images[n_train:n_train+n_val],
        "test": images[n_train+n_val:]
    }

    split_counts[cls] = {}

    for split in splits:

        os.makedirs(f"{TARGET_DIR}/{split}/{cls}", exist_ok=True)
        """
        for img_path in tqdm(
            splits[split],
            desc=f"{cls}-{split}",
            leave=False
        ):
        """

        for img_path in  splits[split]:
            filename = os.path.basename(img_path)
            dst = f"{TARGET_DIR}/{split}/{cls}/{filename}"
            shutil.copy(img_path, dst)

        split_counts[cls][split] = len(splits[split])

Classes: 100%|██████████| 3/3 [21:55<00:00, 438.62s/it]


real test

In [8]:
for cls in real_data:

    os.makedirs(f"{REAL_TEST_DIR}/{cls}", exist_ok=True)

    for img_path in real_data[cls]:
        filename = os.path.basename(img_path)
        dst = f"{REAL_TEST_DIR}/{cls}/{filename}"
        shutil.copy(img_path, dst)

print summary

In [9]:
print("\nSynthetic split:\n")

for cls in split_counts:
    print(f"{cls}:")
    for split in split_counts[cls]:
        print(f"  {split}: {split_counts[cls][split]}")
    print()

print("\nReal test set:")
for cls in real_data:
    print(f"{cls}: {len(real_data[cls])}")


Synthetic split:

clean:
  train: 1354
  val: 290
  test: 291

finished:
  train: 1400
  val: 300
  test: 300

full:
  train: 1400
  val: 300
  test: 300


Real test set:
full: 31
empty: 11
clean: 5
finished_leftovers: 18
